# Plotter workflow
SVG -> split by pen -> optimize -> G-code -> plotter. `profile.test_mode` is on by default, so nothing here touches hardware.

In [ ]:
%load_ext autoreload
%autoreload 2
import plotter_utils as pu

## 1. Machine profile
Edit the values marked unverified in the README, save once, reload next time.

In [ ]:
import os
PROFILE = "machine.json"
profile = pu.MachineProfile.load(PROFILE) if os.path.exists(PROFILE) else pu.MachineProfile()
profile.paper_size = "a1"
# profile.bed_size_mm = (841, 594)   # set once measured; enables bounds checking
profile.save(PROFILE)
profile

## 2. Split the artwork into one SVG per pen

In [ ]:
layers = pu.split_layers("MoulinRouge.svg", group_by_pen=True)   # one file per pen
for l in layers:
    print(l.index, l.pen, l.name, l.filename)

## 3. Build a job and step through it
The job is plain JSON, so you can edit or re-run it later.

In [ ]:
job = pu.build_layer_job(layers, optimize=True)   # paper size comes from the SVG
job.save("jobfiles/MoulinRouge.steps.json")
g = pu.connect(profile)   # simulated while profile.test_mode is True
runner = pu.job_stepper(job, pu.JobContext(profile, connection=g))

## 4. Going live (after checking the README's first-connection checklist)

In [ ]:
# profile.test_mode = False        # only once the plotter is connected and checked
# profile.save(PROFILE)
# g = pu.connect(profile)          # now a real GrblConnection
# print(g.status())
# g.set_zero()                     # current pen position becomes X0 Y0
# runner = pu.job_stepper(job, pu.JobContext(profile, connection=g))